## data from twitter API v2

In [1]:
import requests
import json
from datetime import datetime
import time

In [ ]:
BEARER_TOKEN = "AAAAAAAAAAAAAAAAAAAAAO3Z5QEAAAAArWhWUMNTswj90wiVpYU1liecc%2B4%3DKqugGavHmZiL8yms2QTirjVipM5hLVljszlJKyImyP4KpcZDTX"

# Updated code

In [3]:
def get_algerian_news_tweets(
    max_results=100,
    sort_order="recency",
    start_time=None,
    end_time=None,
    min_retweets=None,
    min_likes=None,
    verified_only=False,
    has_images=False,
    has_videos=False,
    has_links=False,
    custom_keywords=None,
    next_token=None
):
    """
    Fetch tweets related to news in Algerian dialect with advanced filtering
    Returns simplified tweet data with fields: category, author, date, source, content
    
    Args:
        max_results: Number of tweets (10-100 for free tier)
        sort_order: "recency" or "relevancy"
        start_time: ISO 8601 format (e.g., "2025-10-20T00:00:00Z")
        end_time: ISO 8601 format
        min_retweets: Minimum number of retweets
        min_likes: Minimum number of likes
        verified_only: Only show verified accounts
        has_images: Only tweets with images
        has_videos: Only tweets with videos
        has_links: Only tweets with links
        custom_keywords: Additional keywords to search for
        next_token: Pagination token from previous response (optional)
    """
    
    # API endpoint for recent search
    url = "https://api.twitter.com/2/tweets/search/recent"
    
    # Headers with authentication
    headers = {
        "Authorization": f"Bearer {BEARER_TOKEN}",
        "Content-Type": "application/json"
    }
    
    # Build search query for Algerian dialect tweets
    query_parts = [
        "(الجزائر OR دزاير OR dzair OR algerie)",
        "(خبار OR أخبار OR خبر OR عاجل OR الآن OR جديد OR "
        "actualité OR info OR news OR urgent OR breaking OR "
        "اليوم OR today OR هام OR important)",
        "(lang:ar OR lang:fr)",
        "-is:retweet"
    ]
    
    # Add custom keywords if provided
    if custom_keywords:
        query_parts.insert(2, f"({custom_keywords})")
    
    # Add engagement filters
    if min_retweets:
        query_parts.append(f"min_retweets:{min_retweets}")
    
    if min_likes:
        query_parts.append(f"min_faves:{min_likes}")
    
    # Add verified filter
    if verified_only:
        query_parts.append("is:verified")
    
    # Add media filters
    if has_images:
        query_parts.append("has:images")
    
    if has_videos:
        query_parts.append("has:videos")
    
    if has_links:
        query_parts.append("has:links")
    
    query = " ".join(query_parts)
    
    # Request parameters - only request fields we need
    params = {
        "query": query,
        "max_results": max_results,
        "sort_order": sort_order,
        "tweet.fields": "created_at,author_id,text,context_annotations",
        "user.fields": "username",
        "expansions": "author_id"
    }
    
    # Add pagination token if provided
    if next_token:
        params["pagination_token"] = next_token
        print(f"Using pagination token: {next_token[:20]}...")
    
    # Add time filters if provided
    if start_time:
        params["start_time"] = start_time
    
    if end_time:
        params["end_time"] = end_time
    
    try:
        response = requests.get(url, headers=headers, params=params)
        
        # Check if request was successful
        if response.status_code == 200:
            raw_data = response.json()
            
            # Transform to simplified format
            simplified_tweets = transform_to_simplified_format(raw_data)
            
            # Return simplified data with metadata
            return {
                'data': simplified_tweets,
                'meta': raw_data.get('meta', {})
            }
            
        elif response.status_code == 429:
            # Rate limit hit
            reset_time = response.headers.get('x-rate-limit-reset')
            if reset_time:
                reset_datetime = datetime.fromtimestamp(int(reset_time))
                print(f"Rate limit exceeded!")
                print(f"Limit resets at: {reset_datetime}")
                print(f"Wait time: {reset_datetime - datetime.now()}")
            else:
                print("Rate limit exceeded! Please wait 24 hours before trying again.")
            print("\nFree tier limit: 50 requests per 24 hours")
            return None
        else:
            print(f"Error: {response.status_code}")
            print(f"Message: {response.text}")
            return None
            
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None


def transform_to_simplified_format(raw_data):
    """
    Transform Twitter API response to simplified format
    Returns list of tweets with: category, author, date, source, content
    """
    if not raw_data or 'data' not in raw_data:
        return []
    
    tweets = raw_data.get('data', [])
    users = {user['id']: user for user in raw_data.get('includes', {}).get('users', [])}
    
    simplified_tweets = []
    
    for tweet in tweets:
        # Get author username
        author_id = tweet.get('author_id')
        author = users.get(author_id, {}).get('username', 'Unknown')
        
        # Extract category from context annotations if available
        category = extract_category(tweet.get('context_annotations', []))
        
        simplified_tweet = {
            'category': category,
            'author': author,
            'date': tweet.get('created_at', ''),
            'source': 'twitter',
            'content': tweet.get('text', '')
        }
        
        simplified_tweets.append(simplified_tweet)
    
    return simplified_tweets


def extract_category(context_annotations):
    """
    Extract category from Twitter context annotations
    Returns 'news' by default or more specific category if available
    """
    if not context_annotations:
        return 'news'
    
    # Look for domain/entity information
    categories = []
    for annotation in context_annotations:
        domain = annotation.get('domain', {})
        entity = annotation.get('entity', {})
        
        # Check domain name
        if domain.get('name'):
            categories.append(domain['name'])
        
        # Check entity name
        if entity.get('name'):
            categories.append(entity['name'])
    
    # Return first category or 'news' as default
    return categories[0] if categories else 'news'


In [4]:
def save_collected_tweets(tweets, filename, next_token=None, oldest_id=None, newest_id=None):
    """Save collected tweets to JSON file with pagination info"""
    meta = {
        'result_count': len(tweets),
        'collected_at': datetime.now().isoformat()
    }
    
    # Add pagination info if available
    if next_token:
        meta['next_token'] = next_token
    if oldest_id:
        meta['oldest_id'] = oldest_id
    if newest_id:
        meta['newest_id'] = newest_id
    
    data = {
        'data': tweets,
        'meta': meta
    }
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


In [5]:
def display_tweets(data):
    """Display tweets in a readable format"""
    
    if not data or 'data' not in data:
        print("No tweets found or error occurred")
        return
    
    tweets = data.get('data', [])
    
    print(f"\n{'='*80}")
    print(f"Found {len(tweets)} tweets")
    print(f"{'='*80}\n")
    
    for i, tweet in enumerate(tweets, 1):
        print(f"Tweet #{i}")
        print(f"Category: {tweet.get('category', 'N/A')}")
        print(f"Author: @{tweet.get('author', 'Unknown')}")
        print(f"Date: {tweet.get('date', 'N/A')}")
        print(f"Source: {tweet.get('source', 'twitter')}")
        print(f"\nContent:\n{tweet.get('content', '')}")
        print(f"{'-'*80}\n")

In [6]:
def save_to_json(data, filename="algerian_tweets.json"):
    """Save tweets to a JSON file"""
    
    if data:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Tweets saved to {filename}")

In [7]:
data = get_algerian_news_tweets(
    max_results=100, 
)
save_to_json(data, filename="algerian_tweets.json")

Tweets saved to algerian_tweets.json


In [13]:
data = get_algerian_news_tweets(
    max_results=100, 
    next_token="b26v89c19zqg8o3fscn4isktdmh9dkuia2r6v58x0ljel"
)
save_to_json(data, filename="algerian_tweets.1.json")

Using pagination token: b26v89c19zqg8o3fscn4...
Tweets saved to algerian_tweets.1.json


In [18]:
data = get_algerian_news_tweets(
    max_results=100, 
    next_token="b26v89c19zqg8o3fsck8ju88eoigadnbgzht783499jzx"
)
save_to_json(data, filename="algerian_tweets.2.json")

Using pagination token: b26v89c19zqg8o3fsck8...
Tweets saved to algerian_tweets.2.json


In [28]:
data = get_algerian_news_tweets(
    max_results=100, 
    next_token="b26v89c19zqg8o3fsck8jq0a3oid2s6vcj7fc20imbpfh"
)
save_to_json(data, filename="algerian_tweets.3.json")

Using pagination token: b26v89c19zqg8o3fsck8...
Tweets saved to algerian_tweets.3.json


## Merging them into one CSV file

In [34]:
import pandas as pd
import json
import os

# List of JSON files to merge
json_files = [
    'algerian_tweets.json',
    'algerian_tweets.1.json',
    'algerian_tweets.2.json',
    'algerian_tweets.3.json'
]

# Initialize list to store all tweets
all_tweets = []

# Read each JSON file and extract tweets
for file in json_files:
    if os.path.exists(file):
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            tweets = data.get('data', [])
            all_tweets.extend(tweets)
        print(f"Loaded {len(tweets)} tweets from {file}")
    else:
        print(f"File not found: {file}")

print(f"\nTotal tweets collected: {len(all_tweets)}")

Loaded 100 tweets from algerian_tweets.json
Loaded 100 tweets from algerian_tweets.1.json
Loaded 100 tweets from algerian_tweets.2.json
Loaded 99 tweets from algerian_tweets.3.json

Total tweets collected: 399


In [35]:
# Create DataFrame with specified columns
df = pd.DataFrame(all_tweets)

# Add missing columns (title and label) with empty values
df['title'] = ''
df['label'] = ''

# Reorder columns to match the desired structure
columns_order = ['title', 'content', 'author', 'source', 'date','category', 'label']
df = df[columns_order]

# Display first few rows
print("\nFirst 5 rows of the merged dataset:")
print(df.head())
print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


First 5 rows of the merged dataset:
  title                                            content        author  \
0        @KoubaReda نفس السكريبت عند سكان الحظيرة الجزا...  1983_chakiri   
1        🚨🚨🚨 — عاجل:\n\n🏆 المنتخبات المتأهلة إلى كأس ال...   foryou12316   
2        Cette mesure ne concerne que le visa non-immig...     TSAlgerie   
3        @makemororocco Le plus important c’est que l’A...      dafirmus   
4        🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨\n\nعاااااااااااااااااااااااااااجل...       Siiros2   

    source                      date           category label  
0  twitter  2025-11-16T15:41:49.000Z               news        
1  twitter  2025-11-16T15:41:07.000Z              Sport        
2  twitter  2025-11-16T15:41:00.000Z  Business Taxonomy        
3  twitter  2025-11-16T15:39:58.000Z               news        
4  twitter  2025-11-16T15:37:43.000Z              Sport        

DataFrame shape: (399, 7)
Columns: ['title', 'content', 'author', 'source', 'date', 'category', 'label']


In [36]:
# Save to CSV file
output_file = 'algerian_tweets_merged.csv'
df.to_csv(output_file, index=False, encoding='utf-8')

In [37]:
df.head()

,title,content,author,source,date,category,label
0,,@KoubaReda نفس السكريبت عند سكان الحظيرة الجزا...,1983_chakiri,twitter,2025-11-16T15:41:49.000Z,news,
1,,🚨🚨🚨 — عاجل:\n\n🏆 المنتخبات المتأهلة إلى كأس ال...,foryou12316,twitter,2025-11-16T15:41:07.000Z,Sport,
2,,Cette mesure ne concerne que le visa non-immig...,TSAlgerie,twitter,2025-11-16T15:41:00.000Z,Business Taxonomy,
3,,@makemororocco Le plus important c’est que l’A...,dafirmus,twitter,2025-11-16T15:39:58.000Z,news,
4,,🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨\n\nعاااااااااااااااااااااااااااجل...,Siiros2,twitter,2025-11-16T15:37:43.000Z,Sport,


In [38]:
df.category.value_counts()

category
news                         248
Unified Twitter Taxonomy      42
Business Taxonomy             19
Person                        15
Place                         14
Sport                         13
Events [Entity Service]       12
Brand Vertical                10
News Vertical                  9
Sports Team                    8
Sports Event                   4
Entities [Entity Service]      2
Reoccurring Trends             2
Interests and Hobbies          1
Name: count, dtype: int64

In [39]:
df.shape

(399, 7)